In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.colors import LinearSegmentedColormap


In [4]:
models ={
    'Oracle-2 Lite': ['../../models/BTSv2-lite/gentle-sweep-1', 
                      '../../models/BTSv2-lite/comfy-smoke-128',
                      '../../models/BTSv2-lite/deft-monkey-129',
                      '../../models/BTSv2-lite/flowing-feather-130',
                      '../../models/BTSv2-lite/stilted-dawn-131'],
    'Oracle-2': ['../../models/BTSv2/peachy-sweep-4',
                 '../../models/BTSv2/rare-night-119', 
                 '../../models/BTSv2/winter-meadow-120', 
                 '../../models/BTSv2/rural-tree-121',
                 '../../models/BTSv2/resilient-feather-125'],
    'Oracle-2 Omni': ['../../models/BTSv2-pro/woven-sweep-9',
                      '../../models/BTSv2-pro/smooth-haze-141',
                      '../../models/BTSv2-pro/splendid-sun-142',
                      '../../models/BTSv2-pro/sweet-glade-143',
                      '../../models/BTSv2-pro/dazzling-thunder-144']
}

colors = {
    'Oracle-2 Lite': '#994882',
    'Oracle-2': '#008080',
    'Oracle-2 Omni': '#FF6645'
}

markers = {
    'Oracle-2 Lite': 's',
    'Oracle-2': 'D',
    'Oracle-2 Omni': 'o',
}

In [5]:
days = 2**np.arange(0, 11)

In [6]:
def get_data(model_path, day, mode, depth):

    path = f"{model_path}/plots/depth{depth}/cf_{mode}/cf_trigger+{day}.npy"
    data = np.load(path)
    return data

In [31]:
from torchgen import model


def get_model_cf(model_name, mode, depth, day):

    assert mode in ['precision', 'recall']

    model_paths = models[model_name]
    n_classses = 2 if depth == 1 else 7
    model_data = np.zeros((n_classses, n_classses, 5)) 

    for i, model_path in enumerate(model_paths):
        print(f"Processing model: {model_name}, path: {model_path}")
        model_data[:, :, i] = get_data(model_path, day, mode=mode, depth=depth)
    
    return np.mean(model_data, axis=2), np.std(model_data, axis=2)

def plot_cf(model_name, mode, depth, day, savefig=False):

    if depth == 1:
        class_names = ["Persistent", "Transient"]
    elif depth == 2:
        class_names = ['AGN', 'CV', "VarStar", "SN-Ia", "SN-II", "SN-Ib/c", "SLSN-I"]
    
    cf_mean, cf_std = get_model_cf(model_name, mode, depth, day)

    c_list = ["#FFFFFF", colors[model_name]]  # White to the model's color
    my_cmap = LinearSegmentedColormap.from_list("custom_blue", c_list, N=256)

    plt.figure(figsize=(5, 5))
    plt.imshow(cf_mean, cmap=my_cmap, vmin=0, vmax=1)
    # plt.colorbar(label=f'{mode.capitalize()}')
    
    if depth == 1:
        plt.title(f'{model_name}: Trigger+{day} days', fontweight='bold', fontsize='xx-large')
    if depth == 2:
        plt.xlabel('Photometric Class', fontweight='bold', fontsize='xx-large')

    if model == 'Oracle-2 Lite':
        plt.ylabel('Spectroscopic Class', fontweight='bold', fontsize='xx-large')

    for i in range(cf_mean.shape[0]):
        for j in range(cf_mean.shape[1]):
            mean_val = cf_mean[i, j]
            std_val = cf_std[i, j]
            if len(class_names) < 4:
                plt.text(j, i, f'{mean_val:.2f}\n±{std_val:.2f}', ha='center', va='center', color='black', fontsize='x-large')
            else: 
                plt.text(j, i, f'{mean_val:.2f}\n±{std_val:.2f}', ha='center', va='center', color='black')

    # Add class labels to axes
    plt.xticks(ticks=np.arange(len(class_names)), labels=class_names)
    plt.yticks(ticks=np.arange(len(class_names)), labels=class_names, rotation=90, va='center')

    plt.tight_layout()
    if savefig:
        plt.savefig(f'Depth{depth}/{mode}/{model_name}_depth{depth}_day{day}_{mode}.pdf')
        plt.close()
    else:
        plt.show()

In [32]:
for model in models.keys():
    for d in days:
        for mode in ['precision', 'recall']:
            for depth in [1, 2]:
                plot_cf(model, mode, depth, d, savefig=True)

Processing model: Oracle-2 Lite, path: ../../models/BTSv2-lite/gentle-sweep-1
Processing model: Oracle-2 Lite, path: ../../models/BTSv2-lite/comfy-smoke-128
Processing model: Oracle-2 Lite, path: ../../models/BTSv2-lite/deft-monkey-129
Processing model: Oracle-2 Lite, path: ../../models/BTSv2-lite/flowing-feather-130
Processing model: Oracle-2 Lite, path: ../../models/BTSv2-lite/stilted-dawn-131
Processing model: Oracle-2 Lite, path: ../../models/BTSv2-lite/gentle-sweep-1
Processing model: Oracle-2 Lite, path: ../../models/BTSv2-lite/comfy-smoke-128
Processing model: Oracle-2 Lite, path: ../../models/BTSv2-lite/deft-monkey-129
Processing model: Oracle-2 Lite, path: ../../models/BTSv2-lite/flowing-feather-130
Processing model: Oracle-2 Lite, path: ../../models/BTSv2-lite/stilted-dawn-131
Processing model: Oracle-2 Lite, path: ../../models/BTSv2-lite/gentle-sweep-1
Processing model: Oracle-2 Lite, path: ../../models/BTSv2-lite/comfy-smoke-128
Processing model: Oracle-2 Lite, path: ../../